In [ ]:
import sys
sys.path.insert(0, "../../src")

import numpy as np
import jax
import jax.numpy as jnp
from scipy.optimize import least_squares, minimize, differential_evolution
import matplotlib.pyplot as plt

from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)

# ================================================================
# RAW DAC DATA — JEOL ARM-200F magnification look-up table
# Keys: magnification (string), values: dict of lens DAC hex values
# ================================================================
RAW_DAC_HEX = {
    "2000":    {"IL1": "0x4b4b", "IL2": "0x4b2b", "IL3": "0xaf98", "OLf": "0x625a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2500":    {"IL1": "0x4cd7", "IL2": "0x458f", "IL3": "0xb471", "OLf": "0x562a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "3000":    {"IL1": "0x4e0b", "IL2": "0x4046", "IL3": "0xb8fd", "OLf": "0x582a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "4000":    {"IL1": "0x4fcf", "IL2": "0x3657", "IL3": "0xc0d7", "OLf": "0x656a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "5000":    {"IL1": "0x512f", "IL2": "0x2d1d", "IL3": "0xc719", "OLf": "0x6d6a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "6000":    {"IL1": "0x526e", "IL2": "0x2515", "IL3": "0xcccc", "OLf": "0x684a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "8000":    {"IL1": "0x6a31", "IL2": "0x607f", "IL3": "0xadd0", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe2cb"},
    "10000":   {"IL1": "0x6fe4", "IL2": "0x5c11", "IL3": "0xb1c7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xe5a6"},
    "12000":   {"IL1": "0x73f5", "IL2": "0x5925", "IL3": "0xb6fe", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xeb2f"},
    "15000":   {"IL1": "0x7830", "IL2": "0x56ad", "IL3": "0xc026", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf548"},
    "20000":   {"IL1": "0x8057", "IL2": "0x52d6", "IL3": "0xbeb9", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xf9bb"},
    "25000":   {"IL1": "0x8692", "IL2": "0x50e2", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xffff"},
    "30000":   {"IL1": "0x9f22", "IL2": "0x611e", "IL3": "0x9b35", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "40000":   {"IL1": "0xa336", "IL2": "0x611e", "IL3": "0x8ad8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "50000":   {"IL1": "0xa674", "IL2": "0x6498", "IL3": "0x85b7", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "60000":   {"IL1": "0xa810", "IL2": "0x6a48", "IL3": "0x7f59", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "80000":   {"IL1": "0xa8ea", "IL2": "0x7182", "IL3": "0x77d2", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "100000":  {"IL1": "0xa800", "IL2": "0x7d46", "IL3": "0x7205", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "120000":  {"IL1": "0xa7d4", "IL2": "0x7d46", "IL3": "0x6c7f", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "150000":  {"IL1": "0xa761", "IL2": "0x8431", "IL3": "0x65e6", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "200000":  {"IL1": "0xa6c5", "IL2": "0x957b", "IL3": "0x5c4a", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "250000":  {"IL1": "0xa6a8", "IL2": "0x957b", "IL3": "0x552b", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "300000":  {"IL1": "0xa608", "IL2": "0x9ec6", "IL3": "0x4ba8", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "400000":  {"IL1": "0xa584", "IL2": "0xad0c", "IL3": "0x3e16", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "500000":  {"IL1": "0xa51d", "IL2": "0xb944", "IL3": "0x3268", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "600000":  {"IL1": "0xa4d0", "IL2": "0xc3c7", "IL3": "0x2771", "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "800000":  {"IL1": "0xe300", "IL2": "0xe54f", "IL3": "0x478",  "OLf": "0x990a", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1000000": {"IL1": "0xe300", "IL2": "0xbd3a", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1200000": {"IL1": "0xe300", "IL2": "0xcd09", "IL3": "0x1ac9", "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "1500000": {"IL1": "0xe300", "IL2": "0xec00", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
    "2000000": {"IL1": "0xffff", "IL2": "0xf800", "IL3": "0x0",   "OLf": "0x93ca", "OLc": "0xb2fa", "PL1": "0xfa00"},
}

# ================================================================
# Parse into structured arrays
# ================================================================
LENS_NAMES = ["IL1", "IL2", "IL3", "PL1"]
AUX_NAMES  = ["OLf", "OLc"]

def parse_dac_table(raw):
    """Convert hex DAC table -> dict of arrays, assign operating mode labels."""
    mags = sorted(int(k) for k in raw.keys())
    
    data = {"mag": np.array(mags, dtype=float)}
    for lens in LENS_NAMES + AUX_NAMES:
        data[lens] = np.array([int(raw[str(m)][lens], 16) for m in mags], dtype=float)

    # Assign operating modes
    # Mode C: 30k-600k (OLf = 39178, PL1 = 64000)
    modes = []
    for m in mags:
        if m <= 6000: modes.append("A")
        elif m <= 25000: modes.append("B")
        elif m <= 600000: modes.append("C")
        else: modes.append("D")
    data["mode"] = np.array(modes)
    return data

dac = parse_dac_table(RAW_DAC_HEX)

In [ ]:
# ================================================================
# Setup Target Magnifications and Initial Distances
# ================================================================

# Filter for Mode C (Imaging Mode)
mode_c_idx = np.where(dac["mode"] == "C")[0]
mode_c_mag = dac["mag"][mode_c_idx]

# Define Magnification Targets
# The system magnification is M_total = M_obj * M_proj * M_intermediate
# Here we isolate the intermediate lens magnification: M_intermediate = M_total / (M_obj * M_proj)
M_obj = 60
M_proj = 100
MILSystem = mode_c_mag / (M_obj * M_proj)

# Define Initial Geometry (from JEOL Schematic)
# These will be used as the starting point and bounds for the optimizer
z2_proj = 325 * 1e-3  # mm
z1_proj = z2_proj / M_proj

d_saa_to_il1 = 13.00 * 1e-3
d_il1_to_il2 = 52.083 * 1e-3
d_il2_to_il3 = 62.500 * 1e-3
d_il3_to_im = 52.083 * 1e-3 - z1_proj

print("Target Intermediate Magnifications (MILSystem):")
print(MILSystem)


In [ ]:
# ================================================================
# Solver for d_obj_to_il1 and G_c with generalized IL1 model + Monotonicity Constraint
# ================================================================

def lens_f_from_current_linear(I, G_c):
    return 1 / (I**2 * G_c)

def lens_rotation_from_current_linear(I, R_c):
    return R_c * I

def lens_f_from_current_nonlinear(I, G_c, alpha):
    # Lens power P = 1/f = G_c * I^2 + alpha * I^4
    power = G_c * I**2 + alpha * I**4
    if power <= 0: return 1e9 # Avoid division by zero
    return 1 / power

def loss(params):
    # params: [d_obj, d_12, d_23, d_3im, G_c_1, alpha_1, G_c_2, alpha_2, G_c_3, alpha_3]
    d_obj_to_il1, d_il1_to_il2_opt, d_il2_to_il3_opt, d_il3_to_im_opt, G_c_1, alpha_1, G_c_2, alpha_2, G_c_3, alpha_3 = params
    
    total_loss = 0
    calculated_mags = []
    
    for i, mag_target in zip(mode_c_idx, MILSystem):
        # All lenses nonlinear
        f1 = lens_f_from_current_nonlinear(dac["IL1"][i] / 2**16, G_c_1, alpha_1)
        f2 = lens_f_from_current_nonlinear(dac["IL2"][i] / 2**16, G_c_2, alpha_2)
        f3 = lens_f_from_current_nonlinear(dac["IL3"][i] / 2**16, G_c_3, alpha_3)
        
        p1 = propagation_matrix(d_obj_to_il1, xp=np)
        p2 = propagation_matrix(d_il1_to_il2_opt, xp=np)
        p3 = propagation_matrix(d_il2_to_il3_opt, xp=np)
        p4 = propagation_matrix(d_il3_to_im_opt, xp=np)
        
        l1 = lens_matrix(f1, xp=np)
        l2 = lens_matrix(f2, xp=np)
        l3 = lens_matrix(f3, xp=np)
        
        M_total = p4 @ l3 @ p3 @ l2 @ p2 @ l1 @ p1
        A = M_total[0, 0]
        B = M_total[0, 1]
        
        current_mag = abs(A)
        calculated_mags.append(current_mag)
        
        # We want B = 0 (focus)
        # We want abs(A) = mag_target (magnification)
        # Relaxed B constraint slightly
        total_loss += (B * 1000)**2 * 10 + ((current_mag - mag_target) / mag_target)**2 * 10000
        
    # Soft monotonic constraint penalty
    calculated_mags = np.array(calculated_mags)
    diffs = np.diff(calculated_mags)
    violations = diffs[diffs <= 0]
    if len(violations) > 0:
        total_loss += np.sum(violations**2) * 1e6  # Softer penalty
        
    return total_loss

# Run optimization using differential evolution (global optimizer)
bounds = [
    (8e-3, 20e-3),                               # d_obj_to_il1
    (d_il1_to_il2 - 2e-3, d_il1_to_il2 + 2e-3),  # d_il1_to_il2 (+/- 2mm)
    (d_il2_to_il3 - 2e-3, d_il2_to_il3 + 2e-3),  # d_il2_to_il3 (+/- 2mm)
    (d_il3_to_im - 2e-3, d_il3_to_im + 2e-3),    # d_il3_to_im (+/- 2mm)
    (10, 5000), (-10000, 10000),                 # IL1
    (10, 5000), (-10000, 10000),                 # IL2
    (10, 5000), (-10000, 10000)                  # IL3
]
result_de = differential_evolution(loss, bounds, maxiter=500, popsize=15, tol=1e-4, mutation=(0.5, 1.5), recombination=0.7, seed=42)

# Refine with L-BFGS-B
result = minimize(loss, result_de.x, bounds=bounds, method='L-BFGS-B')

d_obj_to_il1_opt, d_il1_to_il2_opt, d_il2_to_il3_opt, d_il3_to_im_opt, G_c_1_opt, alpha_1_opt, G_c_2_opt, alpha_2_opt, G_c_3_opt, alpha_3_opt = result.x

print(f"Optimization Success: {result.success}")
print(f"Optimized d_obj_to_il1: {d_obj_to_il1_opt * 1e3:.4f} mm")
print(f"Optimized d_il1_to_il2: {d_il1_to_il2_opt * 1e3:.4f} mm (nominal: {d_il1_to_il2 * 1e3:.4f} mm)")
print(f"Optimized d_il2_to_il3: {d_il2_to_il3_opt * 1e3:.4f} mm (nominal: {d_il2_to_il3 * 1e3:.4f} mm)")
print(f"Optimized d_il3_to_im:  {d_il3_to_im_opt * 1e3:.4f} mm (nominal: {d_il3_to_im * 1e3:.4f} mm)")
print(f"Optimized G_c (IL1): {G_c_1_opt:.4f}, alpha: {alpha_1_opt:.4f}")
print(f"Optimized G_c (IL2): {G_c_2_opt:.4f}, alpha: {alpha_2_opt:.4f}")
print(f"Optimized G_c (IL3): {G_c_3_opt:.4f}, alpha: {alpha_3_opt:.4f}")

# Let's check the results
print("\nChecking optimized results:")
print(f"{'Target Mag':>12s} | {'Actual Mag':>12s} | {'B (focus error) [mm]':>20s}")
print("-" * 50)

last_mag = 0
for i, mag_target in zip(mode_c_idx, MILSystem):
    f1 = lens_f_from_current_nonlinear(dac["IL1"][i] / 2**16, G_c_1_opt, alpha_1_opt)
    f2 = lens_f_from_current_nonlinear(dac["IL2"][i] / 2**16, G_c_2_opt, alpha_2_opt)
    f3 = lens_f_from_current_nonlinear(dac["IL3"][i] / 2**16, G_c_3_opt, alpha_3_opt)
    print(f1)
    
    p1 = propagation_matrix(d_obj_to_il1_opt, xp=np)
    p2 = propagation_matrix(d_il1_to_il2_opt, xp=np)
    p3 = propagation_matrix(d_il2_to_il3_opt, xp=np)
    p4 = propagation_matrix(d_il3_to_im_opt, xp=np)
    
    l1 = lens_matrix(f1, xp=np)
    l2 = lens_matrix(f2, xp=np)
    l3 = lens_matrix(f3, xp=np)
    
    M_total = p4 @ l3 @ p3 @ l2 @ p2 @ l1 @ p1
    A = M_total[0, 0]
    B = M_total[0, 1]
    
    mag_str = f"{abs(A):>12.2f}"
    # Check monotonicity for display purposes
    if abs(A) < last_mag:
        mag_str += " (!)"
    last_mag = abs(A)
    
    print(f"{mag_target:>12.2f} | {mag_str} | {B * 1e3:>20.4f}")
